# Importando as bibliotecas básicas

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Importando o dataset

In [2]:
dados = pd.read_excel(r'../Aula 5 - Random Forest/Data_Train.xlsx')

Este dataset tem diversos dados de voos com seu preço associado

O objetivo do problema é conseguir prever o preço de um voo através dos dados disponíveis

# Olhando para os dados

In [3]:
dados

,Airline,Date_of_Journey,Source,Destination,Route,Dep_Time,Arrival_Time,Duration,Total_Stops,Additional_Info,Price
0,IndiGo,24/03/2019,Banglore,New Delhi,BLR → DEL,22:20,01:10 22 Mar,2h 50m,non-stop,No info,3897
1,Air India,1/05/2019,Kolkata,Banglore,CCU → IXR → BBI → BLR,05:50,13:15,7h 25m,2 stops,No info,7662
2,Jet Airways,9/06/2019,Delhi,Cochin,DEL → LKO → BOM → COK,09:25,04:25 10 Jun,19h,2 stops,No info,13882
3,IndiGo,12/05/2019,Kolkata,Banglore,CCU → NAG → BLR,18:05,23:30,5h 25m,1 stop,No info,6218
4,IndiGo,01/03/2019,Banglore,New Delhi,BLR → NAG → DEL,16:50,21:35,4h 45m,1 stop,No info,13302
...,...,...,...,...,...,...,...,...,...,...,...
10678,Air Asia,9/04/2019,Kolkata,Banglore,CCU → BLR,19:55,22:25,2h 30m,non-stop,No info,4107
10679,Air India,27/04/2019,Kolkata,Banglore,CCU → BLR,20:45,23:20,2h 35m,non-stop,No info,4145
10680,Jet Airways,27/04/2019,Banglore,Delhi,BLR → DEL,08:20,11:20,3h,non-stop,No info,7229
10681,Vistara,01/03/2019,Banglore,New Delhi,BLR → DEL,11:30,14:10,2h 40m,non-stop,No info,12648


In [4]:
dados.describe()

,Price
count,10683.000000
mean,9087.064121
std,4611.359167
min,1759.000000
25%,5277.000000
50%,8372.000000
75%,12373.000000
max,79512.000000


In [7]:
dados.dtypes

Airline                str
Date_of_Journey        str
Source                 str
Destination            str
Route                  str
Dep_Time               str
Arrival_Time           str
Duration               str
Total_Stops            str
Additional_Info        str
Price                int64
day                float64
month              float64
year               float64
dtype: object

Temos muitas colunas categóricas neste dataset, então vamos tentar quebrar algumas, de forma a construir algumas variáveis numéricas

# Tratamento dos Dados

## Quebrando a Variável de Data

In [6]:
dados['day'] = pd.Series(dtype=int)
dados['month'] = pd.Series(dtype=int)
dados['year'] = pd.Series(dtype=int)

from datetime import datetime


for i in dados.index:
    
    dados.loc[i, 'day'] = datetime.strptime(dados.loc[i, 'Date_of_Journey'], '%d/%m/%Y').day
    dados.loc[i, 'month'] = datetime.strptime(dados.loc[i, 'Date_of_Journey'], '%d/%m/%Y').month
    dados.loc[i, 'year'] = datetime.strptime(dados.loc[i, 'Date_of_Journey'], '%d/%m/%Y').year


In [7]:
dados

,Airline,Date_of_Journey,Source,Destination,Route,Dep_Time,Arrival_Time,Duration,Total_Stops,Additional_Info,Price,day,month,year
0,IndiGo,24/03/2019,Banglore,New Delhi,BLR → DEL,22:20,01:10 22 Mar,2h 50m,non-stop,No info,3897,24.0,3.0,2019.0
1,Air India,1/05/2019,Kolkata,Banglore,CCU → IXR → BBI → BLR,05:50,13:15,7h 25m,2 stops,No info,7662,1.0,5.0,2019.0
2,Jet Airways,9/06/2019,Delhi,Cochin,DEL → LKO → BOM → COK,09:25,04:25 10 Jun,19h,2 stops,No info,13882,9.0,6.0,2019.0
3,IndiGo,12/05/2019,Kolkata,Banglore,CCU → NAG → BLR,18:05,23:30,5h 25m,1 stop,No info,6218,12.0,5.0,2019.0
4,IndiGo,01/03/2019,Banglore,New Delhi,BLR → NAG → DEL,16:50,21:35,4h 45m,1 stop,No info,13302,1.0,3.0,2019.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10678,Air Asia,9/04/2019,Kolkata,Banglore,CCU → BLR,19:55,22:25,2h 30m,non-stop,No info,4107,9.0,4.0,2019.0
10679,Air India,27/04/2019,Kolkata,Banglore,CCU → BLR,20:45,23:20,2h 35m,non-stop,No info,4145,27.0,4.0,2019.0
10680,Jet Airways,27/04/2019,Banglore,Delhi,BLR → DEL,08:20,11:20,3h,non-stop,No info,7229,27.0,4.0,2019.0
10681,Vistara,01/03/2019,Banglore,New Delhi,BLR → DEL,11:30,14:10,2h 40m,non-stop,No info,12648,1.0,3.0,2019.0


In [8]:
dados = dados.drop('Date_of_Journey', axis = 1)

## Quebrando a variável de Hora (Partida e Chegada)

In [10]:
dados['departure_hour'] = pd.Series(dtype=int)
dados['departure_min'] = pd.Series(dtype=int)
dados['arrival_hour'] = pd.Series(dtype=int)
dados['arrival_min'] = pd.Series(dtype=int)


for i in dados.index:
    
    dados.loc[i, 'departure_hour'] = int(dados.loc[i, 'Dep_Time'][0:2])
    dados.loc[i, 'departure_min'] = int(dados.loc[i, 'Dep_Time'][3:6])
    dados.loc[i, 'arrival_hour'] = int(dados.loc[i, 'Arrival_Time'][0:2])
    dados.loc[i, 'arrival_min'] = int(dados.loc[i, 'Arrival_Time'][3:6])


In [11]:
dados

,Airline,Source,Destination,Route,Dep_Time,Arrival_Time,Duration,Total_Stops,Additional_Info,Price,day,month,year,departure_hour,departure_min,arrival_hour,arrival_min
0,IndiGo,Banglore,New Delhi,BLR → DEL,22:20,01:10 22 Mar,2h 50m,non-stop,No info,3897,24.0,3.0,2019.0,22.0,20.0,1.0,10.0
1,Air India,Kolkata,Banglore,CCU → IXR → BBI → BLR,05:50,13:15,7h 25m,2 stops,No info,7662,1.0,5.0,2019.0,5.0,50.0,13.0,15.0
2,Jet Airways,Delhi,Cochin,DEL → LKO → BOM → COK,09:25,04:25 10 Jun,19h,2 stops,No info,13882,9.0,6.0,2019.0,9.0,25.0,4.0,25.0
3,IndiGo,Kolkata,Banglore,CCU → NAG → BLR,18:05,23:30,5h 25m,1 stop,No info,6218,12.0,5.0,2019.0,18.0,5.0,23.0,30.0
4,IndiGo,Banglore,New Delhi,BLR → NAG → DEL,16:50,21:35,4h 45m,1 stop,No info,13302,1.0,3.0,2019.0,16.0,50.0,21.0,35.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10678,Air Asia,Kolkata,Banglore,CCU → BLR,19:55,22:25,2h 30m,non-stop,No info,4107,9.0,4.0,2019.0,19.0,55.0,22.0,25.0
10679,Air India,Kolkata,Banglore,CCU → BLR,20:45,23:20,2h 35m,non-stop,No info,4145,27.0,4.0,2019.0,20.0,45.0,23.0,20.0
10680,Jet Airways,Banglore,Delhi,BLR → DEL,08:20,11:20,3h,non-stop,No info,7229,27.0,4.0,2019.0,8.0,20.0,11.0,20.0
10681,Vistara,Banglore,New Delhi,BLR → DEL,11:30,14:10,2h 40m,non-stop,No info,12648,1.0,3.0,2019.0,11.0,30.0,14.0,10.0


In [12]:
dados = dados.drop('Dep_Time', axis = 1)
dados = dados.drop('Arrival_Time', axis = 1)

In [13]:
dados

,Airline,Source,Destination,Route,Duration,Total_Stops,Additional_Info,Price,day,month,year,departure_hour,departure_min,arrival_hour,arrival_min
0,IndiGo,Banglore,New Delhi,BLR → DEL,2h 50m,non-stop,No info,3897,24.0,3.0,2019.0,22.0,20.0,1.0,10.0
1,Air India,Kolkata,Banglore,CCU → IXR → BBI → BLR,7h 25m,2 stops,No info,7662,1.0,5.0,2019.0,5.0,50.0,13.0,15.0
2,Jet Airways,Delhi,Cochin,DEL → LKO → BOM → COK,19h,2 stops,No info,13882,9.0,6.0,2019.0,9.0,25.0,4.0,25.0
3,IndiGo,Kolkata,Banglore,CCU → NAG → BLR,5h 25m,1 stop,No info,6218,12.0,5.0,2019.0,18.0,5.0,23.0,30.0
4,IndiGo,Banglore,New Delhi,BLR → NAG → DEL,4h 45m,1 stop,No info,13302,1.0,3.0,2019.0,16.0,50.0,21.0,35.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10678,Air Asia,Kolkata,Banglore,CCU → BLR,2h 30m,non-stop,No info,4107,9.0,4.0,2019.0,19.0,55.0,22.0,25.0
10679,Air India,Kolkata,Banglore,CCU → BLR,2h 35m,non-stop,No info,4145,27.0,4.0,2019.0,20.0,45.0,23.0,20.0
10680,Jet Airways,Banglore,Delhi,BLR → DEL,3h,non-stop,No info,7229,27.0,4.0,2019.0,8.0,20.0,11.0,20.0
10681,Vistara,Banglore,New Delhi,BLR → DEL,2h 40m,non-stop,No info,12648,1.0,3.0,2019.0,11.0,30.0,14.0,10.0


Agora que temos algumas variáveis novas na nossa base de dados, vamos tratar as variáveis categóricas

## Tratando a variável de quantidade de paradas

In [14]:
dados.Total_Stops.unique()


<StringArray>
['non-stop', '2 stops', '1 stop', '3 stops', nan, '4 stops']
Length: 6, dtype: str

In [15]:
dados.Total_Stops = dados.Total_Stops.fillna('non-stop')
dados['stops'] = pd.Series(dtype=int)

for i in dados.index:
    if dados.loc[i, 'Total_Stops'] == 'non-stop':
        dados.loc[i, 'stops'] = 0
    else:
        dados.loc[i, 'stops'] = int(dados.loc[i, 'Total_Stops'][0:2])

In [16]:
dados

,Airline,Source,Destination,Route,Duration,Total_Stops,Additional_Info,Price,day,month,year,departure_hour,departure_min,arrival_hour,arrival_min,stops
0,IndiGo,Banglore,New Delhi,BLR → DEL,2h 50m,non-stop,No info,3897,24.0,3.0,2019.0,22.0,20.0,1.0,10.0,0.0
1,Air India,Kolkata,Banglore,CCU → IXR → BBI → BLR,7h 25m,2 stops,No info,7662,1.0,5.0,2019.0,5.0,50.0,13.0,15.0,2.0
2,Jet Airways,Delhi,Cochin,DEL → LKO → BOM → COK,19h,2 stops,No info,13882,9.0,6.0,2019.0,9.0,25.0,4.0,25.0,2.0
3,IndiGo,Kolkata,Banglore,CCU → NAG → BLR,5h 25m,1 stop,No info,6218,12.0,5.0,2019.0,18.0,5.0,23.0,30.0,1.0
4,IndiGo,Banglore,New Delhi,BLR → NAG → DEL,4h 45m,1 stop,No info,13302,1.0,3.0,2019.0,16.0,50.0,21.0,35.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10678,Air Asia,Kolkata,Banglore,CCU → BLR,2h 30m,non-stop,No info,4107,9.0,4.0,2019.0,19.0,55.0,22.0,25.0,0.0
10679,Air India,Kolkata,Banglore,CCU → BLR,2h 35m,non-stop,No info,4145,27.0,4.0,2019.0,20.0,45.0,23.0,20.0,0.0
10680,Jet Airways,Banglore,Delhi,BLR → DEL,3h,non-stop,No info,7229,27.0,4.0,2019.0,8.0,20.0,11.0,20.0,0.0
10681,Vistara,Banglore,New Delhi,BLR → DEL,2h 40m,non-stop,No info,12648,1.0,3.0,2019.0,11.0,30.0,14.0,10.0,0.0


In [17]:
dados = dados.drop('Total_Stops', axis = 1)

# Aplicando o Label Encoder

In [18]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

colunas = dados.dtypes.reset_index()

categ_cols = colunas[colunas[0] == 'str']['index'].to_list()


# Preenchendo as variáveis com dados nulos com a média


for i in categ_cols:
    dados[str(i) +'_encoded'] = le.fit_transform(dados[i])
    dados = dados.drop(i,axis = 1)


In [19]:
dados

,Price,day,month,year,departure_hour,departure_min,arrival_hour,arrival_min,stops,Airline_encoded,Source_encoded,Destination_encoded,Route_encoded,Duration_encoded,Additional_Info_encoded
0,3897,24.0,3.0,2019.0,22.0,20.0,1.0,10.0,0.0,3,0,5,18,240,8
1,7662,1.0,5.0,2019.0,5.0,50.0,13.0,15.0,2.0,1,3,0,84,336,8
2,13882,9.0,6.0,2019.0,9.0,25.0,4.0,25.0,2.0,4,2,1,118,106,8
3,6218,12.0,5.0,2019.0,18.0,5.0,23.0,30.0,1.0,3,3,0,91,311,8
4,13302,1.0,3.0,2019.0,16.0,50.0,21.0,35.0,1.0,3,0,5,29,303,8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10678,4107,9.0,4.0,2019.0,19.0,55.0,22.0,25.0,0.0,0,3,0,64,236,8
10679,4145,27.0,4.0,2019.0,20.0,45.0,23.0,20.0,0.0,1,3,0,64,237,8
10680,7229,27.0,4.0,2019.0,8.0,20.0,11.0,20.0,0.0,4,0,2,18,280,8
10681,12648,1.0,3.0,2019.0,11.0,30.0,14.0,10.0,0.0,10,0,5,18,238,8


# Quebrando o dataset para modelagem

In [20]:
from sklearn.model_selection import train_test_split

x = dados.drop('Price', axis = 1)
y = dados['Price']

x_train,x_test,y_train,y_test = train_test_split(x,y, test_size=0.3, random_state=42)

# Carregando o modelo

Iremos fazer o mesmo processo para os 3 modelos que falamos para verificar as diferenças

In [21]:
%pip install xgboost

Note: you may need to restart the kernel to use updated packages.


In [22]:
%pip install lightgbm

Note: you may need to restart the kernel to use updated packages.


In [23]:
from sklearn.ensemble import GradientBoostingRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.metrics import r2_score

import time

modelo_gr = GradientBoostingRegressor(max_depth=10, max_leaf_nodes=20, random_state=42)
modelo_xb = XGBRegressor(max_depth=10, max_leaf_nodes=20, random_state=42)
modelo_lg = LGBMRegressor(max_depth=10, max_leaf_nodes=20, random_state=42)

print('=========== Gradient Boosting ==============')

start = time.time()
modelo_gr.fit(x_train, y_train)
end = time.time()
y_pred_train = modelo_gr.predict(x_train)
y_pred_test = modelo_gr.predict(x_test)

print('O resultado na base de treino é: ',r2_score(y_train,y_pred_train))
print('O resultado na base de teste é: ',r2_score(y_test,y_pred_test))
print('O tempo que o modelo demorou para treinar foi: ', str(end-start))

print('=========== XG Boost ==============')

start = time.time()
modelo_xb.fit(x_train, y_train)
end = time.time()
y_pred_train = modelo_xb.predict(x_train)
y_pred_test = modelo_xb.predict(x_test)


print('O resultado na base de treino é: ',r2_score(y_train,y_pred_train))
print('O resultado na base de teste é: ',r2_score(y_test,y_pred_test))
print('O tempo que o modelo demorou para treinar foi: ', str(end-start))


print('=========== LGBM ==============')

start = time.time()
modelo_lg.fit(x_train, y_train)
end = time.time()
y_pred_train = modelo_lg.predict(x_train)
y_pred_test = modelo_lg.predict(x_test)

print('O resultado na base de treino é: ',r2_score(y_train,y_pred_train))
print('O resultado na base de teste é: ',r2_score(y_test,y_pred_test))
print('O tempo que o modelo demorou para treinar foi: ', str(end-start))


=========== Gradient Boosting ==============
O resultado na base de treino é:  0.9348709064319582
O resultado na base de teste é:  0.8939378418121372
O tempo que o modelo demorou para treinar foi:  0.8409304618835449
=========== XG Boost ==============


c:\Git\study-python-ml\.venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [15:56:39] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:794: 
Parameters: { "max_leaf_nodes" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


O resultado na base de treino é:  0.9959851503372192
O resultado na base de teste é:  0.8877706527709961
O tempo que o modelo demorou para treinar foi:  0.5086345672607422
=========== LGBM ==============
[LightGBM] [Warning] num_leaves is set=31, max_leaf_nodes=20 will be ignored. Current value: num_leaves=31
[LightGBM] [Warning] num_leaves is set=31, max_leaf_nodes=20 will be ignored. Current value: num_leaves=31
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000173 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 438
[LightGBM] [Info] Number of data points in the train set: 7478, number of used features: 13
[LightGBM] [Info] Start training from score 9121.404654
[LightGBM] [Warning] num_leaves is set=31, max_leaf_nodes=20 will be ignored. Current value: num_leaves=31
[LightGBM] [Warning] num_leaves is set=31, max_leaf_nodes=20 will be

É possível ver as principais diferenças entre os modelos

- O GradientBoosting é um bom parâmetro para início

- O XGBoost traz resultados mais precisos

- O LGBM é treinado com uma velocidade muito maior

# Fazendo a Feature Selection

## Select KBest

In [24]:
from sklearn.feature_selection import SelectKBest, f_regression

In [25]:
selector = SelectKBest(score_func=f_regression, k = 7)

selector.fit(x,y)

dados_selected = selector.transform(x)

In [26]:
dados_selected

array([[ 24.,   3.,  10., ...,   5.,  18., 240.],
       [  1.,   5.,  15., ...,   0.,  84., 336.],
       [  9.,   6.,  25., ...,   1., 118., 106.],
       ...,
       [ 27.,   4.,  20., ...,   2.,  18., 280.],
       [  1.,   3.,  10., ...,   5.,  18., 238.],
       [  9.,   5.,  15., ...,   1., 108., 347.]], shape=(10683, 7))

In [27]:
cols = selector.get_support(indices=True)
dados_new_best = x.iloc[:,cols]

Estas são as 7 colunas (features) que mais influenciam no preço das passagens

## Select Percentile

In [28]:
from sklearn.feature_selection import SelectPercentile, f_regression


selector = SelectPercentile(score_func=f_regression,percentile=50)
selector.fit(x,y)
dados_selected = selector.transform(x)


cols = selector.get_support(indices=True)
dados_new_percentile = x.iloc[:,cols]

In [29]:
dados_new_percentile

,day,month,arrival_min,stops,Destination_encoded,Route_encoded,Duration_encoded
0,24.0,3.0,10.0,0.0,5,18,240
1,1.0,5.0,15.0,2.0,0,84,336
2,9.0,6.0,25.0,2.0,1,118,106
3,12.0,5.0,30.0,1.0,0,91,311
4,1.0,3.0,35.0,1.0,5,29,303
...,...,...,...,...,...,...,...
10678,9.0,4.0,25.0,0.0,0,64,236
10679,27.0,4.0,20.0,0.0,0,64,237
10680,27.0,4.0,20.0,0.0,2,18,280
10681,1.0,3.0,10.0,0.0,5,18,238


## Rodando Novamente os Modelos com a Feature Selection Realizada

### Select KBest

In [30]:
x_train,x_test,y_train,y_test = train_test_split(dados_new_best,y, test_size=0.3, random_state=42)

In [31]:
from sklearn.ensemble import GradientBoostingRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.metrics import r2_score

import time

modelo_gr = GradientBoostingRegressor(max_depth=10, max_leaf_nodes=20, random_state=42)
modelo_xb = XGBRegressor(max_depth=10, max_leaf_nodes=20, random_state=42)
modelo_lg = LGBMRegressor(max_depth=10, max_leaf_nodes=20, random_state=42)

print('=========== Gradient Boosting ==============')

start = time.time()
modelo_gr.fit(x_train, y_train)
end = time.time()
y_pred_train = modelo_gr.predict(x_train)
y_pred_test = modelo_gr.predict(x_test)

print('O resultado na base de treino é: ',r2_score(y_train,y_pred_train))
print('O resultado na base de teste é: ',r2_score(y_test,y_pred_test))
print('O tempo que o modelo demorou para treinar foi: ', str(end-start))

print('=========== XG Boost ==============')

start = time.time()
modelo_xb.fit(x_train, y_train)
end = time.time()
y_pred_train = modelo_xb.predict(x_train)
y_pred_test = modelo_xb.predict(x_test)


print('O resultado na base de treino é: ',r2_score(y_train,y_pred_train))
print('O resultado na base de teste é: ',r2_score(y_test,y_pred_test))
print('O tempo que o modelo demorou para treinar foi: ', str(end-start))


print('=========== LGBM ==============')

start = time.time()
modelo_lg.fit(x_train, y_train)
end = time.time()
y_pred_train = modelo_lg.predict(x_train)
y_pred_test = modelo_lg.predict(x_test)

print('O resultado na base de treino é: ',r2_score(y_train,y_pred_train))
print('O resultado na base de teste é: ',r2_score(y_test,y_pred_test))
print('O tempo que o modelo demorou para treinar foi: ', str(end-start))


=========== Gradient Boosting ==============
O resultado na base de treino é:  0.8491028168273105
O resultado na base de teste é:  0.7251947558491539
O tempo que o modelo demorou para treinar foi:  0.6092493534088135
=========== XG Boost ==============


c:\Git\study-python-ml\.venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [15:57:14] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:794: 
Parameters: { "max_leaf_nodes" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


O resultado na base de treino é:  0.9596468210220337
O resultado na base de teste é:  0.6729627847671509
O tempo que o modelo demorou para treinar foi:  0.325700044631958
=========== LGBM ==============
[LightGBM] [Warning] num_leaves is set=31, max_leaf_nodes=20 will be ignored. Current value: num_leaves=31
[LightGBM] [Warning] num_leaves is set=31, max_leaf_nodes=20 will be ignored. Current value: num_leaves=31
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000072 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 355
[LightGBM] [Info] Number of data points in the train set: 7478, number of used features: 7
[LightGBM] [Info] Start training from score 9121.404654
[LightGBM] [Warning] num_leaves is set=31, max_leaf_nodes=20 will be ignored. Current value: num_leaves=31
[LightGBM] [Warning] num_leaves is set=31, max_leaf_nodes=20 will be i

### Select Percentile

In [32]:
x_train,x_test,y_train,y_test = train_test_split(dados_new_percentile,y, test_size=0.3, random_state=42)


from sklearn.ensemble import GradientBoostingRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.metrics import r2_score

import time

modelo_gr = GradientBoostingRegressor(max_depth=10, max_leaf_nodes=20, random_state=42)
modelo_xb = XGBRegressor(max_depth=10, max_leaf_nodes=20, random_state=42)
modelo_lg = LGBMRegressor(max_depth=10, max_leaf_nodes=20, random_state=42)

print('=========== Gradient Boosting ==============')

start = time.time()
modelo_gr.fit(x_train, y_train)
end = time.time()
y_pred_train = modelo_gr.predict(x_train)
y_pred_test = modelo_gr.predict(x_test)

print('O resultado na base de treino é: ',r2_score(y_train,y_pred_train))
print('O resultado na base de teste é: ',r2_score(y_test,y_pred_test))
print('O tempo que o modelo demorou para treinar foi: ', str(end-start))

print('=========== XG Boost ==============')

start = time.time()
modelo_xb.fit(x_train, y_train)
end = time.time()
y_pred_train = modelo_xb.predict(x_train)
y_pred_test = modelo_xb.predict(x_test)


print('O resultado na base de treino é: ',r2_score(y_train,y_pred_train))
print('O resultado na base de teste é: ',r2_score(y_test,y_pred_test))
print('O tempo que o modelo demorou para treinar foi: ', str(end-start))


print('=========== LGBM ==============')

start = time.time()
modelo_lg.fit(x_train, y_train)
end = time.time()
y_pred_train = modelo_lg.predict(x_train)
y_pred_test = modelo_lg.predict(x_test)

print('O resultado na base de treino é: ',r2_score(y_train,y_pred_train))
print('O resultado na base de teste é: ',r2_score(y_test,y_pred_test))
print('O tempo que o modelo demorou para treinar foi: ', str(end-start))


=========== Gradient Boosting ==============
O resultado na base de treino é:  0.8491028168273105
O resultado na base de teste é:  0.7251947558491539
O tempo que o modelo demorou para treinar foi:  0.5552306175231934
=========== XG Boost ==============


c:\Git\study-python-ml\.venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [15:57:20] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:794: 
Parameters: { "max_leaf_nodes" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


O resultado na base de treino é:  0.9596468210220337
O resultado na base de teste é:  0.6729627847671509
O tempo que o modelo demorou para treinar foi:  0.34003567695617676
=========== LGBM ==============
[LightGBM] [Warning] num_leaves is set=31, max_leaf_nodes=20 will be ignored. Current value: num_leaves=31
[LightGBM] [Warning] num_leaves is set=31, max_leaf_nodes=20 will be ignored. Current value: num_leaves=31
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000078 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 355
[LightGBM] [Info] Number of data points in the train set: 7478, number of used features: 7
[LightGBM] [Info] Start training from score 9121.404654
[LightGBM] [Warning] num_leaves is set=31, max_leaf_nodes=20 will be ignored. Current value: num_leaves=31
[LightGBM] [Warning] num_leaves is set=31, max_leaf_nodes=20 will be

# Cross Validation

Pegando o modelo inicial que performou melhor, vamos testar fazer o cross validation

In [39]:
from sklearn.model_selection import train_test_split

x = dados.drop('Price', axis = 1)
y = dados['Price']

x_train,x_test,y_train,y_test = train_test_split(x,y, test_size=0.3, random_state=42)

modelo_gb = GradientBoostingRegressor(max_depth=10, max_leaf_nodes=20, random_state=42)

print('=========== Gradient Boosting ==============')

start = time.time()
modelo_gb.fit(x_train, y_train)
end = time.time()
y_pred_train = modelo_gb.predict(x_train)
y_pred_test = modelo_gb.predict(x_test)


print('O resultado na base de treino é: ',r2_score(y_train,y_pred_train))
print('O resultado na base de teste é: ',r2_score(y_test,y_pred_test))
print('O tempo que o modelo demorou para treinar foi: ', str(end-start))

=========== Gradient Boosting ==============
O resultado na base de treino é:  0.9348709064319582
O resultado na base de teste é:  0.8939378418121372
O tempo que o modelo demorou para treinar foi:  0.9305977821350098


In [34]:
from sklearn.model_selection import GridSearchCV


parametros = {'n_estimators':[10, 50, 100],
              'learning_rate':[0.1 ,1 ,10],
              'max_depth': [10,50 ,500],
              'max_leaf_nodes':[10,50, 100]}

grid = GridSearchCV(estimator = modelo_gb,
                      param_grid = parametros,
                      cv = 5,
                      scoring = 'r2')



grid.fit(x,y)


c:\Git\study-python-ml\.venv\Lib\site-packages\sklearn\model_selection\_search.py:1248: RuntimeWarning: overflow encountered in square
  (array - array_means[:, np.newaxis]) ** 2, axis=1, weights=weights


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",GradientBoost...ndom_state=42)
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'learning_rate': [0.1, 1, ...], 'max_depth': [10, 50, ...], 'max_leaf_nodes': [10, 50, ...], 'n_estimators': [10, 50, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'r2'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"ver

In [35]:
grid.best_params_


{'learning_rate': 0.1,
 'max_depth': 10,
 'max_leaf_nodes': 50,
 'n_estimators': 100}

In [36]:
grid.best_score_

np.float64(0.8984440229008035)

In [38]:
melhor_modelo = grid.best_estimator_
melhor_modelo

,"max_depth max_depth: int or None, default=3Maximum depth of the individual regression estimators. The maximumdepth limits the number of nodes in the tree. Tune this parameterfor best performance; the best value depends on the interactionof the input variables. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.If int, values must be in the range `[1, inf)`.",10
,"random_state random_state: int, RandomState instance or None, default=NoneControls the random seed given to each Tree estimator at eachboosting iteration.In addition, it controls the random permutation of the features ateach split (see Notes for more details).It also controls the random splitting of the training data to obtain avalidation set if `n_iter_no_change` is not None.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",42
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.Values must be in the range `[2, inf)`.If None, then unlimited number of leaf nodes.",50
,"loss loss: {'squared_error', 'absolute_error', 'huber', 'quantile'}, default='squared_error'Loss function to be optimized. 'squared_error' refers to the squarederror for regression. 'absolute_error' refers to the absolute error ofregression and is a robust loss function. 'huber' is acombination of the two. 'quantile' allows quantile regression (use`alpha` to specify the quantile).See:ref:`sphx_glr_auto_examples_ensemble_plot_gradient_boosting_quantile.py`for an example that demonstrates quantile regression for creatingprediction intervals with `loss='quantile'`.",'squared_error'
,"learning_rate learning_rate: float, default=0.1Learning rate shrinks the contribution of each tree by `learning_rate`.There is a trade-off between learning_rate and n_estimators.Values must be in the range `[0.0, inf)`.",0.1
,"n_estimators n_estimators: int, default=100The number of boosting stages to perform. Gradient boostingis fairly robust to over-fitting so a large number usuallyresults in better performance.Values must be in the range `[1, inf)`.",100
,"subsample subsample: float, default=1.0The fraction of samples to be used for fitting the individual baselearners. If smaller than 1.0 this results in Stochastic GradientBoosting. `subsample` interacts with the parameter `n_estimators`.Choosing `subsample < 1.0` leads to a reduction of varianceand an increase in bias.Values must be in the range `(0.0, 1.0]`.",1.0
,"criterion criterion: {'friedman_mse', 'squared_error'}, default='friedman_mse'This parameter has no effect... versionadded:: 0.18.. deprecated:: 1.9 `criterion` is deprecated and will be removed in 1.11.",'deprecated'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, values must be in the range `[2, inf)`.- If float, values must be in the range `(0.0, 1.0]` and `min_samples_split` will be `ceil(min_samples_split * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, values must be in the range `[1, inf)`.- If float, values must be in the range `(0.0, 1.0)` and `min_samples_leaf` will be `ceil(min_samples_leaf * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.V